# Remote-Photoplethysmography (RPPG) and Respiration Signal Processing 
Created by : Dinda Joycehana_122140048

#### Setup Depedencies

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

### Membuat Model MediaPipe

MediaPipe External Models was downloaded from GitHub: Informatics Engineering Sumatera Institute of Technology

Credit : 
https://github.com/informatika-itera/if3024-handson/tree/main/2025-notebooks/models


In [2]:
## Define the models face detector
base_model="models/blaze_face_short_range.tflite"
base_options = python.BaseOptions(model_asset_path=base_model)

## Mediapipe configuration options
FaceDetectorOptions = vision.FaceDetectorOptions
VisionRunningMode = vision.RunningMode
options = FaceDetectorOptions(
    base_options=base_options,
    running_mode = VisionRunningMode.IMAGE,
)
face_detector = vision.FaceDetector.create_from_options(options)

In [3]:
## Define the models Landmaark Pose
model_path = "models/pose_landmarker.task"
BaseOptions = mp.tasks.BaseOptions

PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options_image = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=model_path,
    ),
    running_mode=VisionRunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)

pose_landmarker = vision.PoseLandmarker.create_from_options(options_image)

### **RPPG: POS (Plane-Orthogonal to Skin)**

Metode rPPG mengambil sinyal denyut dari tubuh berdasarkan analisis perubahan warna yang dipancarkan kulit, yang kemudian diubah menjadi sinyal fisiologis.

In [4]:
# Create POS Function
def POS(signal, **kargs):
    eps = 10**-9
    X = signal
    e, c, f = X.shape # Nilai dari estimator, channel (RGB), dan frame
    w = int(1.6 * kargs['fps']) # Window length in frames

    #P: Matriks transformasi warna dari RGB ke proyeksi POS
    #Q: Matriks p akan diklon berulang sebanyak e kali 
   
    P = np.array([[0, 1, -1], [-2, 1, 1]])
    Q = np.stack([P for _ in range(e)], axis = 0)

    ## H: Matriks keluaran yang akan menyimpan hasil dari setiap frame
    H = np.zeros((e, f))
    for n in np.arange(w, f): #looping sliding window
        m = n - w + 1

        #Normalisasi sinyal dari pengaruh cahaya
        Cn = X[:, :, m:(n+1)]
        M = 1.0 / (np.mean(Cn, axis = 2) + eps)
        M = np.expand_dims(M, axis=2) # shape [e, c, w]
        Cn = np.multiply(Cn, M)

       #proyeksikan sinyal RGB ke dalam domain POS
        S = np.dot(Q, Cn)
        S = S[0, :, :, :]
        S = np.swapaxes(S, 0, 1) 

       #Tuning sinyal
        S1 = S[:, 0, :]
        S2 = S[:, 1, :]
        alpha = np.std(S1, axis=1) / (eps + np.std(S2, axis=1))
        alpha = np.expand_dims(alpha, axis=1)
        Hn = np.add(S1, alpha * S2)
        Hnm = Hn - np.expand_dims(np.mean(Hn, axis=1), axis=1)

        #Overlap-add
        H[:, m:(n + 1)] = np.add(H[:, m:(n + 1)], Hnm)  # Add the tuned signal to the output matrix

    return H

### **Respiration : ROI and Features**

In [20]:
## Membuat fungsi untuk mendapatkan ROI awal berdasarkan posisi bahu
#def get_initial_roi(image, landmarker, x_size = 140, y_size=100, shift_x=0, shift_y=0): #Mmengatur jarak roi dari sisi kanan dan kiri frame
#    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#    height, width = image.shape[:2]
#
#    ## Create Mediapipe image
#    mp_image = mp.Image(
#        image_format = mp.ImageFormat.SRGB,
#        data = image_rgb
#    )
#
#    ## Inference Detect Landmarks
#    detection_result = landmarker.detect(mp_image)
#
#    if not detection_result.pose_landmarks:
#        raise ValueError("No pose detected in the frame")
#    
#    landmarks = detection_result.pose_landmarks[0]
#
#    ## Mengatur posisi landmark bahu kiri dan kanan
#    left_shoulder = landmarks[11]
#    right_shoulder = landmarks[12]
#
#    ## Calculate the center point between should before creating the ROI Bounds
#    center_x = int((left_shoulder.x + right_shoulder.x) * width / 2)
#    center_y = int((left_shoulder.y + right_shoulder.y) * height / 2)
#
#    ## Apply shift to the center point (offset from the chest)
#    center_x += shift_x
#    center_y += shift_y
#
#    ## Create the ROI Bounds from the center point and sizes
#    left_x = max(0, center_x - x_size)
#    right_x = min(width, center_x + x_size)
#    top_y = max(0, center_y - y_size)
#    bottom_y = min(height, center_y)
#
#    ## Validate ROI size
#    if (right_x - left_x) <= 0 or (bottom_y - top_y) <= 0:
#        raise ValueError("Invalid ROI dimensions")
#    
#    ## Return top, left, bottom, right points
#    return (left_x, top_y, right_x, bottom_y)
#
#STANDARD_SIZE = (640, 480)  # Ukuran standar untuk resize frame
##Membuat fungsi untuk mengambil fitur
#def initialize_features(frame):
#    global features, lk_params, old_gray, left_x, top_y, right_x, bottom_y, STANDARD_SIZE   # Declare them as global to modify the outer variables
#
#    roi_coords = get_initial_roi(frame, pose_landmarker)
#    left_x, top_y, right_x, bottom_y = roi_coords
#    
#    frame = cv2.resize(frame, STANDARD_SIZE)
#    old_frame = frame.copy()
#    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
#    roi_chest = old_gray[top_y:bottom_y, left_x:right_x]
#
#    #Mengubah nilai dari parameter goodFeaturesToTrack
#    features = cv2.goodFeaturesToTrack(roi_chest, maxCorners=130, qualityLevel=0.01, minDistance=3, blockSize=3)
#    if features is None:
#        raise ValueError("No features found to track!")
#    features = np.float32(features)
#    features[:,:,0] += left_x
#    features[:,:,1] += top_y
#    old_gray = old_gray
#    lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

In [ ]:
# Initialize variables
fps = 15                    # atur frame rate sesuai dengan kualitas webcam 
time_window = 60            #Panjang durasi video yang dianalisis
frames = time_window * fps  #Jumlah maksimal frame
count = 0                   # jumlah frame yang sudah diproses

#Initialize parameters for face detection 
r_signal, g_signal, b_signal = [], [], [] # menyimpan nilai R, G, dan B dari kulit wajah
margin_x = 10           # beri padding secara horizontal
scaling_factor = 0.8    # mengatur ukuran bounding box

# Initialize variables for tracking
resp_signals = []   # List to store respiration signals
features = None     # features for optical flow tracking
lk_params = None    # Algortihm Lucas-Kanade for optical flow
old_gray = None     # Greyscale image for optical flow tracking
left_x, top_y, right_x, bottom_y = None, None, None, None # Bounding box coordinates
STANDARD_SIZE = (640, 480)  # Ukuran standar untuk resize frame

In [22]:
# Helper functions
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data)
    return y


### **Main Program : RPPG and Respiration**



In [ ]:
try: # set up webcam capture
    cap = cv2.VideoCapture(0)

    fps = 15
    frames = 900  # ~60s at 15fps
    count = 0

    rppg_signals = []
    resp_signals = []
    # Counting the number of frames to capture 60s or 900 frames
    while count < frames:
        ret, frame = cap.read()  #read frame from webcam
        if not ret:
            print("Error : Failed to capture video")
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ## Seting up the Mediapipe Image
        mp_image = mp.Image(
            image_format = mp.ImageFormat.SRGB,
            data = rgb_frame
        )

        ## Process the frame using the face detector
        results = face_detector.detect(mp_image)
         # Pose Landmarks for Shoulder
        pose_result = pose_landmarker.detect(mp_image) 
        frame_height, frame_width, _ = rgb_frame.shape

        if results.detections:
            for detection in results.detections:

                ## Get the Bounding box
                bboxC = detection.bounding_box
                x, y, w, h = bboxC.origin_x, bboxC.origin_y, bboxC.width, bboxC.height

                ## Setup tipis tipis biar boxnya pas di tengah wajah
                new_x = int(x + margin_x)
                new_w = int(w * scaling_factor)
                new_h = int(h * scaling_factor)

                # Cegah keluar batas
                new_x = max(0, min(new_x, frame_width - 1))
                new_y = max(0, min(y, frame_height - 1))
                new_w = min(new_w, frame_width - new_x)                
                new_h = min(new_h, frame_height - new_y)

                ## Get the ROI
                face_roi = rgb_frame [new_y:new_y+new_h, new_x:new_x+new_w]

                ## Calculate the Mean RGB values of the face ROI
                mean_rgb = cv2.mean(face_roi)[:3]

                # Append the combined RGB values to the respective lists
                r_signal.append(mean_rgb[0])
                g_signal.append(mean_rgb[1])
                b_signal.append(mean_rgb[2])       

    
        if pose_result.pose_landmarks:
            landmarks = pose_result.pose_landmarks[0]
            ## Mengatur posisi landmark bahu kiri dan kanan
            left_shoulder = landmarks[11]
            right_shoulder = landmarks[12]
            ## Calculate the center point between should before creating the ROI Bounds
            center_x = int((left_shoulder.x + right_shoulder.x) * frame_width / 2)
            center_y = int((left_shoulder.y + right_shoulder.y) * frame_height / 2)
            ## Apply shift to the center point (offset from the chest)
            center_x += 0
            center_y += 0
            ## Create the ROI Bounds from the center point and sizes
            left_x = max(0, center_x - 140)
            right_x = min(frame_width, center_x + 140)
            top_y = max(0, center_y - 100)
            bottom_y = min(frame_height, center_y)
            ## Validate ROI size
            if (right_x - left_x) <= 0 or (bottom_y - top_y) <= 0:
                raise ValueError("Invalid ROI dimensions")
            # Draw rectangles for visualization
            cv2.rectangle(frame, (int(x), int(y)), (int(x + new_w), int(y + new_h)), (0, 255, 0), 2)
            cv2.rectangle(frame, (int(left_x), int(top_y)), (int(right_x), int(bottom_y)), (0, 255, 0), 2) 
            break  # Only process first face
        cv2.imshow('Webcam', frame)
        if cv2.waitKey(1) & 0xFF == ord('q') :
            break
        count += 1

    
except Exception as e:
    print(f"An error occurred: {e}")
    cap.release()
    cv2.destroyAllWindows()

An error occurred: name 'margin_x' is not defined


In [24]:
# Menghitung rPPG menggunakan Metode POS
rgb_signals = np.array([r_signal, g_signal, b_signal])
rgb_signals = rgb_signals.reshape(1, 3, -1)
rppg_signal = POS(rgb_signals, fps=fps)
rppg_signal = rppg_signal.reshape(-1)


In [3]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.framework.formats import image as mp_image
import matplotlib.pyplot as plt

# --- Setup Face Detector ---
base_model = "models/blaze_face_short_range.tflite"
base_options = python.BaseOptions(model_asset_path=base_model)
FaceDetectorOptions = vision.FaceDetectorOptions
VisionRunningMode = vision.RunningMode
face_options = FaceDetectorOptions(
    base_options=base_options,
    running_mode=VisionRunningMode.IMAGE,
)
face_detector = vision.FaceDetector.create_from_options(face_options)

# --- Setup Pose Landmarker ---
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5)

# --- Webcam Loop ---
cap = cv2.VideoCapture(0)
fps = 15
frames = 900  # ~60s at 15fps
count = 0

rppg_signals = []
shoulder_y = []

while count < frames:
    ret, frame = cap.read()
    if not ret:
        print("Webcam is not available.")
        break

    # Convert to RGB and mp.Image
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp_image(image_format=mp_image.ImageFormat.SRGB, data=rgb)

    # Face detection (bounding box for rPPG)
    face_results = face_detector.detect(mp_img)
    h, w, _ = frame.shape
    if face_results.detections:
        for detection in face_results.detections:
            bbox = detection.bounding_box
            x1 = max(0, int(bbox.origin_x))
            y1 = max(0, int(bbox.origin_y))
            x2 = min(w, int(bbox.origin_x + bbox.width))
            y2 = min(h, int(bbox.origin_y + bbox.height))
            roi_rppg = frame[y1:y2, x1:x2]
            if roi_rppg.size > 0:
                mean_rgb = np.mean(np.mean(roi_rppg, axis=0), axis=0)
                rppg_signals.append(mean_rgb)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            break  # Only process first face

    # Pose landmarks (shoulders for respiration)
    pose_results = pose.process(rgb)
    if pose_results.pose_landmarks:
        left_shoulder = pose_results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_SHOULDER]
        right_shoulder = pose_results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_SHOULDER]
        avg_shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
        shoulder_y.append(avg_shoulder_y)
        lx, ly = int(left_shoulder.x * w), int(left_shoulder.y * h)
        rx, ry = int(right_shoulder.x * w), int(right_shoulder.y * h)
        cv2.circle(frame, (lx, ly), 5, (255,0,0), -1)
        cv2.circle(frame, (rx, ry), 5, (255,0,0), -1)

    cv2.imshow('Webcam', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    count += 1

cap.release()
cv2.destroyAllWindows()

# --- Signal Processing ---
rppg_signals = np.array(rppg_signals)
shoulder_y = np.array(shoulder_y)

# Extract green channel for rPPG
if len(rppg_signals) > 0:
    green_signal = rppg_signals[:, 1]
else:
    green_signal = np.array([])

# Helper functions for filtering
from scipy.signal import butter, filtfilt, find_peaks

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data)
    return y

# Filter rPPG (heart rate: 0.7-4.0 Hz)
if len(green_signal) > 0:
    filtered_rppg = bandpass_filter(green_signal, 0.7, 4.0, fps)
    rppg_peaks, _ = find_peaks(filtered_rppg, distance=int(fps/4))
    time_rppg = np.arange(len(filtered_rppg)) / fps
    heart_rate = len(rppg_peaks) * 60 / time_rppg[-1]
else:
    filtered_rppg = np.array([])
    rppg_peaks = []
    time_rppg = np.array([])
    heart_rate = 0

# Filter shoulder signal (respiration: 0.1-0.4 Hz)
if len(shoulder_y) > 0:
    filtered_resp = bandpass_filter(shoulder_y, 0.1, 0.4, fps)
    resp_peaks, _ = find_peaks(filtered_resp, distance=int(fps/2))
    time_resp = np.arange(len(filtered_resp)) / fps
    resp_rate = len(resp_peaks) * 60 / time_resp[-1]
else:
    filtered_resp = np.array([])
    resp_peaks = []
    time_resp = np.array([])
    resp_rate = 0

# --- Plot Results ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

# Plot rPPG signal
if len(time_rppg) > 0:
    ax1.plot(time_rppg, filtered_rppg, 'b-', label='Filtered rPPG')
    ax1.plot(time_rppg[rppg_peaks], filtered_rppg[rppg_peaks], 'ro', label='Peaks')
    ax1.set_title(f'rPPG Signal (Heart Rate: {heart_rate:.1f} BPM)')
    ax1.set_xlabel('Time (s)')
    ax1.legend()
else:
    ax1.set_title('No rPPG data')

# Plot respiration signal
if len(time_resp) > 0:
    ax2.plot(time_resp, filtered_resp, 'g-', label='Filtered Respiration')
    ax2.plot(time_resp[resp_peaks], filtered_resp[resp_peaks], 'ro', label='Peaks')
    ax2.set_title(f'Respiration Signal (Rate: {resp_rate:.1f} breaths/min)')
    ax2.set_xlabel('Time (s)')
    ax2.legend()
else:
    ax2.set_title('No respiration data')

plt.tight_layout()
plt.show()

ImportError: cannot import name 'image' from 'mediapipe.framework.formats' (c:\TUGAS KULIAH DINDA\Semester 6\DSP\TugasBesar_DSP_122140048\.venv\lib\site-packages\mediapipe\framework\formats\__init__.py)